In [ ]:
!pip -q install transformers accelerate timm opencv-python pillow

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
model_id = "IDEA-Research/grounding-dino-base"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id)

In [ ]:
uploaded = files.upload()

image_name = list(uploaded.keys())[0]

image = Image.open(image_name).convert("RGB")

image

In [ ]:
text = "person. car. bus. truck. bicycle. motorcycle. dog."

In [ ]:
inputs = processor(
    images=image,
    text=text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    input_ids=inputs.input_ids,
    threshold=0.35,
    text_threshold=0.25,
    target_sizes=[image.size[::-1]]
)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
help(processor.post_process_grounded_object_detection)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)

for score, label, box in zip(
    results[0]["scores"],
    results[0]["labels"],
    results[0]["boxes"]
):
    x1, y1, x2, y2 = box.tolist()

    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )

    ax.add_patch(rect)

    ax.text(
        x1,
        y1,
        f"{label} {score:.2f}",
        color="white",
        bbox=dict(facecolor="red")
    )

plt.axis("off")
plt.show()

In [ ]:
from google.colab import files

uploaded = files.upload()

video_name = list(uploaded.keys())[0]
print(video_name)

In [ ]:
import cv2
import os
from PIL import Image
import numpy as np

In [ ]:
os.makedirs("output_frames", exist_ok=True)

In [ ]:
cap = cv2.VideoCapture(video_name)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps}")
print(f"Resolution: {width} x {height}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
!pip install -q transformers
!pip install -q accelerate
!pip install -q timm
!pip install -q supervision
!pip install -q opencv-python
!pip install -q pillow
!pip install -q matplotlib

In [ ]:
import cv2
import torch
import numpy as np

from PIL import Image
from tqdm import tqdm

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

In [ ]:
model_id = "IDEA-Research/grounding-dino-base"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForZeroShotObjectDetection.from_pretrained(
    model_id
).to(device)

print("Grounding DINO Loaded Successfully")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
video_path = "demo.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Frames:", total_frames)
print("Resolution:", width, "x", height)

In [ ]:
output_path = "GroundingDINO_Output.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

video_writer = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

In [ ]:
TEXT_PROMPT = (
    "person . "
    "car . "
    "truck . "
    "bus . "
    "motorcycle . "
    "bicycle . "
    "bag . "
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
TEXT_PROMPT = "person . car . bus . truck . motorcycle . bicycle . bag ."

In [ ]:
video_path = "demo.mp4"

In [ ]:
output_path = "GroundingDINO_Output.mp4"

In [ ]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

video_writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from tqdm import tqdm
from PIL import Image
import cv2
import torch

FRAME_SKIP = 4

cap = cv2.VideoCapture(video_path)

frame_no = 0

with tqdm(total=total_frames) as pbar:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_no % FRAME_SKIP != 0:
            frame_no += 1
            pbar.update(1)
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(rgb)

        inputs = processor(
            images=image,
            text=TEXT_PROMPT,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        target_sizes = torch.tensor(
            [[image.height, image.width]],
            device=device
        )

        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            threshold=0.35,
            text_threshold=0.25,
            target_sizes=target_sizes
        )

        result = results[0]

        boxes = result["boxes"].cpu().numpy()
        scores = result["scores"].cpu().numpy()
        labels = result["labels"]

        for box, score, label in zip(boxes, scores, labels):

            x1, y1, x2, y2 = map(int, box)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                2
            )

            cv2.putText(
                frame,
                f"{label} {score:.2f}",
                (x1, max(20, y1-10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

        video_writer.write(frame)

        frame_no += 1
        pbar.update(1)

cap.release()
video_writer.release()

print("✅ Detection Completed Successfully!")

In [ ]:
import cv2

cap = cv2.VideoCapture("GroundingDINO_Output.mp4")

print("Opened:", cap.isOpened())
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

cap.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture("GroundingDINO_Output.mp4")

ret, frame = cap.read()

cap.release()

print("Frame read:", ret)

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10,6))
    plt.imshow(frame)
    plt.axis("off")

In [ ]:
from google.colab import files

files.download("GroundingDINO_Output.mp4")

In [ ]:
!pip install -q supervision
!pip install -q onemetric

In [ ]:
import supervision as sv

In [ ]:
import cv2
import torch
import numpy as np
import supervision as sv
from PIL import Image

# -----------------------------
# Input / Output
# -----------------------------
INPUT_VIDEO = "demo.mp4"
OUTPUT_VIDEO = "GroundingDINO_ByteTrack.mp4"

# -----------------------------
# Open video
# -----------------------------
cap = cv2.VideoCapture(INPUT_VIDEO)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

video_writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# ByteTrack

tracker = sv.ByteTrack()

box_annotator = sv.BoxAnnotator()

label_annotator = sv.LabelAnnotator()

frame_number = 0

# Detection Classes

TEXT_PROMPT = (
    "person. car. bus. truck. bicycle. motorcycle. bag. "
)

# Process Video

while True:

    ret, frame = cap.read()

    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    image = Image.fromarray(rgb)

    inputs = processor(
        images=image,
        text=TEXT_PROMPT,
        return_tensors="pt"
    )

    inputs = {k:v.to(device) for k,v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs=outputs,
        input_ids=inputs["input_ids"],
        threshold=0.35,
        text_threshold=0.25,
        target_sizes=[image.size[::-1]]
    )

    result = results[0]

    boxes = result["boxes"].cpu().numpy()

    scores = result["scores"].cpu().numpy()

    labels = result["labels"]

    if len(boxes)==0:

        video_writer.write(frame)

        continue

    detections = sv.Detections(
        xyxy=boxes,
        confidence=scores,
        class_id=np.arange(len(labels))
    )

    detections = tracker.update_with_detections(
        detections
    )

    tracker_labels=[]

    for tracker_id,label in zip(
        detections.tracker_id,
        labels
    ):

        tracker_labels.append(
            f"{label} ID:{tracker_id}"
        )

    frame = box_annotator.annotate(
        scene=frame,
        detections=detections
    )

    frame = label_annotator.annotate(
        scene=frame,
        detections=detections,
        labels=tracker_labels
    )

    video_writer.write(frame)

    frame_number+=1

    if frame_number%20==0:

        print(f"Processed {frame_number} frames")

cap.release()

video_writer.release()

print("Done!")

In [ ]:
from IPython.display import Video

Video("GroundingDINO_ByteTrack.mp4", embed=True)

In [ ]:
from google.colab import files

files.download("GroundingDINO_ByteTrack.mp4")